# Harvard's Endowment: Section 2 (Mean-Variance Optimization)
Data: `multi_asset_etf_data.xlsx`, sheet `excess returns` (monthly, excess over SHV). QAI is dropped, leaving 10 assets. No inflation adjustment.
Annualization: mean x 12, vol x sqrt(12), applied to the statistics (not the raw returns).

In [2]:
import numpy as np, pandas as pd
pd.options.display.float_format = '{:.4f}'.format

X = pd.read_excel('multi_asset_etf_data.xlsx', sheet_name='excess returns', index_col=0)
R = X.drop(columns='QAI')           # 10 assets
mu, Sigma = R.mean(), R.cov()       # monthly sample moments
print(R.shape, R.index.min().date(), '->', R.index.max().date())

(186, 10) 2011-02-28 -> 2026-07-31


## 2.1 Summary statistics

In [3]:
stats = pd.DataFrame({'mean': mu * 12,
                      'vol': R.std() * np.sqrt(12)})
stats['sharpe'] = stats['mean'] / stats['vol']
stats = stats.sort_values('sharpe', ascending=False)
display(stats)
print('Best Sharpe :', stats.sharpe.idxmax(), round(stats.sharpe.max(), 3))
print('Worst Sharpe:', stats.sharpe.idxmin(), round(stats.sharpe.min(), 3))

,mean,vol,sharpe
SPY,0.1270,0.1414,0.8979
HYG,0.0345,0.0733,0.4705
EFA,0.0638,0.1486,0.4291
IYR,0.0701,0.1655,0.4235
PSP,0.0764,0.2113,0.3617
TIP,0.0128,0.0497,0.2568
EEM,0.0444,0.1791,0.2481
IEF,0.0086,0.0621,0.1384
DBC,0.0130,0.1723,0.0755
BWX,-0.0172,0.0819,-0.2096


Best Sharpe : SPY 0.898
Worst Sharpe: BWX -0.21


**Answer.** SPY has the best Sharpe ratio (0.90), almost double the runner-up HYG (0.47). BWX has the worst (-0.21); it is the only asset with a negative mean excess return.

## 2.2 Descriptive analysis

In [4]:
C = R.corr()
display(C.style.format('{:.2f}').background_gradient(cmap='RdBu_r', vmin=-1, vmax=1))

pairs = C.where(np.triu(np.ones(C.shape, dtype=bool), k=1)).stack()
print('Highest:', pairs.idxmax(), round(pairs.max(), 3))
print('Lowest :', pairs.idxmin(), round(pairs.min(), 3))

,BWX,DBC,EEM,EFA,HYG,IEF,IYR,PSP,SPY,TIP
BWX,1.00,0.16,0.61,0.61,0.60,0.58,0.56,0.52,0.44,0.68
DBC,0.16,1.00,0.44,0.44,0.42,-0.31,0.25,0.42,0.38,0.08
EEM,0.61,0.44,1.00,0.81,0.67,0.05,0.57,0.70,0.69,0.38
EFA,0.61,0.44,0.81,1.00,0.78,0.06,0.70,0.86,0.83,0.40
HYG,0.60,0.42,0.67,0.78,1.00,0.19,0.73,0.80,0.78,0.54
IEF,0.58,-0.31,0.05,0.06,0.19,1.00,0.32,0.01,0.01,0.76
IYR,0.56,0.25,0.57,0.70,0.73,0.32,1.00,0.73,0.74,0.60
PSP,0.52,0.42,0.70,0.86,0.80,0.01,0.73,1.00,0.87,0.40
SPY,0.44,0.38,0.69,0.83,0.78,0.01,0.74,0.87,1.00,0.39
TIP,0.68,0.08,0.38,0.40,0.54,0.76,0.60,0.40,0.39,1.00


Highest: ('PSP', 'SPY') 0.872
Lowest : ('DBC', 'IEF') -0.309


In [5]:
# How have TIPS (TIP) done against domestic bonds (IEF) and foreign bonds (BWX)?
display(stats.loc[['TIP', 'IEF', 'BWX']])

,mean,vol,sharpe
TIP,0.0128,0.0497,0.2568
IEF,0.0086,0.0621,0.1384
BWX,-0.0172,0.0819,-0.2096


**Answer.** Highest correlation: PSP and SPY (0.87). Lowest: DBC and IEF (-0.31).

TIPS beat both bond benchmarks: higher mean (1.28% vs 0.86% for IEF and -1.72% for BWX), lower volatility than both (4.97% vs 6.21% and 8.19%), and a higher Sharpe (0.26 vs 0.14 and -0.21). In absolute terms the performance is still modest.

## 2.3 The MV frontier

In [6]:
def tangency(mu, Sigma):
    w = np.linalg.solve(Sigma, mu)       # Sigma^-1 mu
    return pd.Series(w / w.sum(), index=mu.index)

def stats_of(w, mu, Sigma):
    m = w @ mu * 12
    v = np.sqrt(w @ Sigma @ w * 12)
    return pd.Series({'mean': m, 'vol': v, 'sharpe': m / v})

w_tan = tangency(mu, Sigma)
tab = pd.DataFrame({'w_tan': w_tan, 'sharpe': stats['sharpe']})
tab['weight_rank'] = tab['w_tan'].rank(ascending=False).astype(int)
tab['sharpe_rank'] = tab['sharpe'].rank(ascending=False).astype(int)
display(tab.sort_values('w_tan', ascending=False))
print('Spearman rank corr (weights vs Sharpe):', round(tab['w_tan'].corr(tab['sharpe'], method='spearman'), 3))
print('Weights sum to', round(w_tan.sum(), 6))
stats_of(w_tan, mu, Sigma).to_frame('tangency')

,w_tan,sharpe,weight_rank,sharpe_rank
IEF,1.3188,0.1384,1,8
SPY,1.2632,0.8979,2,1
HYG,0.2038,0.4705,3,2
EFA,0.1848,0.4291,4,3
DBC,0.0363,0.0755,5,9
EEM,-0.0010,0.2481,6,7
TIP,-0.1920,0.2568,7,6
IYR,-0.2237,0.4235,8,4
PSP,-0.4106,0.3617,9,5
BWX,-1.1797,-0.2096,10,10


Spearman rank corr (weights vs Sharpe): 0.382
Weights sum to 1.0


,tangency
mean,0.1617
vol,0.1096
sharpe,1.4758


**Answer.** No, the weight ranking does not follow the Sharpe ranking. IEF gets the largest weight (1.32) with a Sharpe of only 0.14, because it is nearly uncorrelated with equities and negatively correlated with DBC, so it hedges. TIP, IYR and PSP are shorted despite positive Sharpe ratios because they are largely spanned by assets already held (TIP by IEF, corr 0.76; IYR and PSP by SPY, corr 0.74 and 0.87). BWX is shorted (-1.18) and is the only asset with a negative Sharpe.

Tangency portfolio: mean 16.17%, vol 10.96%, Sharpe 1.476.

## 2.4 TIPS

In [7]:
R_ex = R.drop(columns='TIP')
w_ex = tangency(R_ex.mean(), R_ex.cov())

mu_up = mu.copy(); mu_up['TIP'] += 0.0012
w_up = tangency(mu_up, Sigma)

W = pd.concat({'base': w_tan, 'no TIPS': w_ex, 'TIPS +0.0012': w_up}, axis=1)
display(W)

perf = pd.DataFrame({
    'base':         stats_of(w_tan, mu, Sigma),
    'no TIPS':      stats_of(w_ex, R_ex.mean(), R_ex.cov()),
    'TIPS +0.0012': stats_of(w_up, mu_up, Sigma),
})
display(perf)

# Is the 0.0012 adjustment large relative to estimation error?
se_ann = R['TIP'].std() * np.sqrt(12) / np.sqrt(len(R) / 12)
print('TIP annual mean:', round(mu['TIP'] * 12, 4), '| std. error of that mean:', round(se_ann, 4),
      '| adjustment (annual):', round(0.0012 * 12, 4), '| in SEs:', round(0.0012 * 12 / se_ann, 2))

,base,no TIPS,TIPS +0.0012
BWX,-1.1797,-1.1375,-0.8602
DBC,0.0363,0.0258,-0.0433
EEM,-0.0010,-0.0062,-0.0402
EFA,0.1848,0.1881,0.2094
HYG,0.2038,0.1791,0.0172
IEF,1.3188,1.1616,0.1289
IYR,-0.2237,-0.2234,-0.2216
PSP,-0.4106,-0.4004,-0.3338
SPY,1.2632,1.2128,0.8819
TIP,-0.1920,NaN,1.2617


,base,no TIPS,TIPS +0.0012
mean,0.1617,0.1554,0.1327
vol,0.1096,0.1054,0.0842
sharpe,1.4758,1.4745,1.5749


TIP annual mean: 0.0128 | std. error of that mean: 0.0126 | adjustment (annual): 0.0144 | in SEs: 1.14


**Answer.**
- *Dropping TIPS* barely matters: Sharpe goes from 1.476 to 1.475 and the other weights move little. In this sample TIPS add almost nothing the other assets do not already provide.
- *Raising TIP's mean by 0.0012/month* (1.44%/yr) changes the portfolio a lot: the TIP weight goes from -0.19 to +1.26, IEF falls from 1.32 to 0.13, and Sharpe rises to 1.575. But the adjustment is only about one standard error of TIP's sample mean, so it is within estimation noise.

**Conclusion.** On the historical data TIPS do not expand the opportunity set, so the data alone do not justify a separate asset class. The case for one has to come from forward-looking return assumptions and Harvard's real-return objective, which this nominal analysis does not capture.